# 00 Setup and data (Step 1)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aminah9945/dengue-climate-reconstruction/blob/main/notebooks/00_setup_and_data.ipynb)

This notebook prepares the inputs for all other notebooks:

1. clones and installs this repository (in Colab);
2. downloads the **data** into `DATA_DIR` if the processed files are missing: weekly dengue notifications from the InfoDengue API and daily NASA POWER climate for 300 Brazilian municipalities, then builds the tidy tables (`dengue_recon.data`). The download is resumable; expect 30-45 minutes from scratch;
3. fetches the base paper's cross-validation mask (Lim et al. 2026) at a pinned commit. Only the data file is read; none of that repository's code is executed;
4. shows the data-foundation EDA.

Nothing large is stored in the repository: data are always re-downloaded from the original sources (see `data/README.md`).

In [ ]:
# ---- Setup: works in Google Colab and locally (VS Code / Jupyter) ----
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/aminah9945/dengue-climate-reconstruction"
BRANCH = "main"          # use "official-run" if the pull request is not merged yet
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/dengue-climate-reconstruction")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[all]"], check=True)
    # Optional: keep the downloaded data on Google Drive so it survives the session.
    USE_DRIVE = False
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ.setdefault("DATA_DIR", "/content/drive/MyDrive/dengue_recon_data")
else:
    # Local: this notebook lives in <repo>/notebooks/
    REPO = Path(os.environ.get("DENGUE_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()

os.environ.setdefault("DENGUE_ROOT", str(REPO))
os.environ.setdefault("DATA_DIR", str(REPO / "data"))
os.environ.setdefault("RESULTS_DIR", str(REPO / "results"))
os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=1")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))      # makes the package importable without a kernel restart

from dengue_recon import config as C
C.ensure_dirs()
print("repo       :", C.ROOT)
print("DATA_DIR   :", C.DATA_DIR, "| processed data present:", (C.DATA_DIR / "dengue_weekly.parquet").exists())
print("RESULTS_DIR:", C.RESULTS_DIR)

In [ ]:
import json
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200, "display.max_columns", 30)


def show(path, width=900):
    p = Path(path)
    p = p if p.is_absolute() else C.FIG_DIR / p
    display(Image(str(p), width=width)) if p.exists() else print("missing figure:", p)


def have_data():
    ok = (C.DATA_DIR / "dengue_weekly.parquet").exists()
    if not ok:
        print("Processed data not found in DATA_DIR - run notebook 00 first (or set DATA_DIR).")
    return ok

## 1. Data (InfoDengue + NASA POWER + IBGE)

Set `DOWNLOAD = True` to fetch and build the data if it is missing. The steps are those of `python scripts/run_pipeline.py --step data`. 2024 is downloaded but never used by the pilot (`RUN_FINAL_2024 = False`). The cell also builds the El Niño index (ONI, a few seconds), which the official runs in notebook 06 need.

In [ ]:
DOWNLOAD = True
NEEDED = ["dengue_weekly.parquet", "climate_weekly.parquet", "municipalities.csv", "year_summary.csv"]
missing = [f for f in NEEDED if not (C.DATA_DIR / f).exists()]
print("missing processed files:", missing or "none")
if missing and DOWNLOAD:
    from dengue_recon.data import build_dataset, download, eda
    download.main(["infodengue", "--top-pop", "700"])
    download.main(["select", "--n", "300"])
    download.main(["power"])
    build_dataset.main()
    eda.main()          # EDA figures and the smoke-test baselines (needed for sanity check 7)
if not (C.DATA_DIR / "oni_weekly.parquet").exists() and DOWNLOAD:
    from dengue_recon.data import oni
    oni.main()          # El Nino index (NOAA CPC, a few seconds); needed by the official runs (notebook 06)
if have_data():
    d = pd.read_parquet(C.DATA_DIR / "dengue_weekly.parquet", columns=["geocode", "epi_year", "cases"])
    print(d.geocode.nunique(), "municipalities;", d.epi_year.min(), "-", d.epi_year.max(), ";", len(d), "weekly rows")

## 2. Base-paper data (Lim et al. 2026 CV mask, read as data only)

In [ ]:
from dengue_recon.base_paper.fetch import fetch_lim_mask
mask = fetch_lim_mask()
print("mask:", mask, mask.exists())
pd.read_csv(C.LIM_TARGET_TABLE).head(3)

## 3. Data-foundation EDA (precomputed in `results/data_foundation/`)

In [ ]:
for f in ["seasonal_curves_by_region.png", "peak_week_variation.png", "coverage_heatmap.png"]:
    show(C.REPORTS_DIR / "figures" / f)
pd.read_csv(C.REPORTS_DIR / "baseline_smoke_test_summary.csv", header=[0, 1], index_col=0)

Setup done. Continue with `01_reproduce_base_paper.ipynb`.